In [25]:
import pandas as pd
import joblib 
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [26]:
# load dataset
dataset = pd.read_csv("../../data/processed/California_Housing.csv")

In [27]:
# Normalize

target = "median_income"
x = dataset.drop(columns=target)
x["ocean_proximity"] = pd.factorize(x["ocean_proximity"])[0]
y = dataset[target]

scaler = StandardScaler()

x_train ,x_test , y_train , y_test = train_test_split(x,y,test_size=0.25 , random_state=42)

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

X_train_tensor = torch.tensor(x_train , dtype=torch.float32)
Y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(x_test , dtype=torch.float32)
Y_test_tensor  = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [32]:
#traing process
model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters() , lr = 0.001)

for epoch in range(500):
    y_pred = model(X_train_tensor)
    
    loss = criterion(y_pred, Y_train_tensor)  # Y_train_tensor = float32
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 18.45047378540039
Epoch 20, Loss: 15.089700698852539
Epoch 40, Loss: 9.285346984863281
Epoch 60, Loss: 4.503335475921631
Epoch 80, Loss: 3.111207962036133
Epoch 100, Loss: 2.3239543437957764
Epoch 120, Loss: 1.9788306951522827
Epoch 140, Loss: 1.8169516324996948
Epoch 160, Loss: 1.7048654556274414
Epoch 180, Loss: 1.6039936542510986
Epoch 200, Loss: 1.507361650466919
Epoch 220, Loss: 1.4148311614990234
Epoch 240, Loss: 1.325771689414978
Epoch 260, Loss: 1.2405240535736084
Epoch 280, Loss: 1.1604007482528687
Epoch 300, Loss: 1.0879167318344116
Epoch 320, Loss: 1.0253360271453857
Epoch 340, Loss: 0.9737738370895386
Epoch 360, Loss: 0.9249598979949951
Epoch 380, Loss: 0.8866084814071655
Epoch 400, Loss: 0.8576216101646423
Epoch 420, Loss: 0.8351908326148987
Epoch 440, Loss: 0.8176252245903015
Epoch 460, Loss: 0.8033112287521362
Epoch 480, Loss: 0.7905106544494629


In [33]:
#Evaluation
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    mse = criterion(y_test_pred, Y_test_tensor)
    rmse = torch.sqrt(mse)
    print("Test RMSE:", rmse.item())
    
print(torch.isnan(Y_train_tensor).sum())  # ถ้า >0 → มี NaN
print(torch.isinf(Y_train_tensor).sum())  # ถ้า >0 → มี Inf
print(torch.isnan(X_train_tensor).sum())  # input ก็ต้อง check

Test RMSE: nan
tensor(0)
tensor(0)
tensor(0)
